In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

# Silver Transformation — UK Bank Holidays

Transform GOV.UK bank-holiday snapshots into structured holiday records for England and Wales.

This notebook:

1. Reads Bronze bank-holiday snapshots.
2. Parses JSON with an explicit schema.
3. Extracts England and Wales holiday events.
4. Converts dates to Spark `date`.
5. Applies data-quality checks.
6. Creates deterministic holiday keys.
7. Deduplicates repeated source snapshots.
8. Merges validated holidays into Silver.

**Source:** `workspace.urbanpulse_bronze.bank_holidays`

**Target:** `workspace.urbanpulse_silver.bank_holidays`

## 1. Initialise project paths

In [0]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if not SRC_PATH.exists():
    raise FileNotFoundError(
        f"Source directory not found: {SRC_PATH}"
    )

if str(SRC_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_PATH),
    )

print(f"Project root: {PROJECT_ROOT}")

## 2. Import transformation components

In [0]:
from pyspark.sql import functions as F

from urbanpulse.transformations.bank_holidays import (
    transform_bank_holidays,
)

from urbanpulse.quality.bank_holidays import (
    valid_bank_holidays,
    invalid_bank_holidays,
)

from urbanpulse.utils.delta import (
    merge_insert_only,
)

## 3. Define source and target tables

In [0]:
BRONZE_TABLE = (
    "workspace."
    "urbanpulse_bronze."
    "bank_holidays"
)

SILVER_TABLE = (
    "workspace."
    "urbanpulse_silver."
    "bank_holidays"
)

## 4. Read Bronze snapshots

In [0]:
bronze_df = spark.table(
    BRONZE_TABLE
)

print(
    f"Bronze holiday snapshots: "
    f"{bronze_df.count()}"
)

display(
    bronze_df.select(
        "request_id",
        "ingested_at",
        "http_status",
    )
    .orderBy(
        F.col("ingested_at").desc()
    )
)

## 5. Parse holiday events

Parse the raw GOV.UK response and explode the England and Wales `events` array into one row per holiday.

In [0]:
parsed_df = transform_bank_holidays(
    bronze_df
)

print(
    f"Parsed holiday rows: "
    f"{parsed_df.count()}"
)

display(
    parsed_df
    .orderBy("holiday_date")
)

## 6. Inspect parsed fields

In [0]:
display(
    parsed_df.select(
        "division",
        "holiday_name",
        "holiday_date",
        "notes",
        "bunting",
        "snapshot_at",
    )
    .orderBy("holiday_date")
)

## 7. Apply data-quality rules

Valid holiday records require:

- request ID
- division
- holiday name
- parsed holiday date
- snapshot timestamp

In [0]:
valid_df = valid_bank_holidays(
    parsed_df
)

invalid_df = invalid_bank_holidays(
    parsed_df
)

valid_count = valid_df.count()
invalid_count = invalid_df.count()

print(
    f"Valid holidays: {valid_count}"
)

print(
    f"Invalid holidays: {invalid_count}"
)

## 8. Enforce the holiday data contract

In [0]:
if invalid_count > 0:
    display(invalid_df)

    raise ValueError(
        f"Data quality failure: "
        f"{invalid_count} invalid "
        "bank-holiday records"
    )

print(
    "Bank-holiday quality checks passed."
)

## 9. Create the holiday key

A deterministic SHA-256 key identifies each holiday independently of the source snapshot that delivered it.

In [0]:
keyed_df = (
    valid_df
    .withColumn(
        "holiday_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("division"),
                F.col(
                    "holiday_date"
                ).cast("string"),
                F.col("holiday_name"),
            ),
            256,
        ),
    )
)

## 10. Deduplicate repeated holiday snapshots

Repeated source snapshots of the same holiday are collapsed to one Silver record.

The most recently collected version is retained.

In [0]:
from pyspark.sql.window import Window


holiday_window = (
    Window
    .partitionBy(
        "holiday_key"
    )
    .orderBy(
        F.col("snapshot_at").desc()
    )
)


deduplicated_df = (
    keyed_df
    .withColumn(
        "_row_number",
        F.row_number().over(
            holiday_window
        ),
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

print(
    f"Rows before deduplication: "
    f"{valid_count}"
)

print(
    f"Unique holidays: "
    f"{deduplicated_df.count()}"
)

## 11. Add Silver processing metadata

In [0]:
silver_df = (
    deduplicated_df
    .withColumn(
        "processed_at",
        F.current_timestamp(),
    )
)

## 12. Verify holiday-key uniqueness

In [0]:
duplicate_keys_df = (
    silver_df
    .groupBy("holiday_key")
    .count()
    .filter(
        F.col("count") > 1
    )
)

if duplicate_keys_df.count() > 0:
    display(duplicate_keys_df)

    raise ValueError(
        "Duplicate holiday keys detected."
    )

print(
    "Holiday keys are unique."
)

## 13. Merge holidays into Silver

Holiday reference data is upserted so later source corrections can update existing Silver records.

In [0]:
from urbanpulse.utils.delta import (
    merge_upsert,
)

In [0]:
merge_result = merge_upsert(
    spark=spark,
    source_df=silver_df,
    target_table=SILVER_TABLE,
    merge_condition="""
        target.holiday_key
        =
        source.holiday_key
    """,
)

print(
    f"Silver bank-holiday table: "
    f"{merge_result}"
)

## 14. Verify Silver bank holidays

In [0]:
%sql
SELECT
    holiday_key,
    division,
    holiday_date,
    holiday_name,
    notes,
    bunting,
    snapshot_at,
    processed_at
FROM workspace.urbanpulse_silver.bank_holidays
ORDER BY holiday_date;

In [0]:
%sql
SELECT
    holiday_key,
    COUNT(*) AS records
FROM workspace.urbanpulse_silver.bank_holidays
GROUP BY holiday_key
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT
    holiday_date,
    COUNT(*) AS holidays,
    COLLECT_LIST(
        holiday_name
    ) AS holiday_names
FROM workspace.urbanpulse_silver.bank_holidays
GROUP BY holiday_date
HAVING COUNT(*) > 1
ORDER BY holiday_date;

In [0]:
%sql
SELECT
    YEAR(holiday_date) AS holiday_year,
    COUNT(*) AS holidays
FROM workspace.urbanpulse_silver.bank_holidays
GROUP BY YEAR(holiday_date)
ORDER BY holiday_year;

In [0]:
%sql
SELECT COUNT(*) AS holidays
FROM workspace.urbanpulse_silver.bank_holidays;

In [0]:
%sql
SELECT
    COUNT(*) AS bronze_snapshots
FROM workspace.urbanpulse_bronze.bank_holidays;

In [0]:
%sql
SELECT
    COUNT(*) AS silver_holidays
FROM workspace.urbanpulse_silver.bank_holidays;